# 10 · VUP coverage-gap model

**Question.** Among high-need households, which ones does VUP currently reach, and which profiles are relatively underserved?

| | |
|---|---|
| **Population** | High-need households: the top quintile of the Cost-of-Living Distress Score **excluding its VUP component** (notebook 07). Sensitivity: the top two quintiles. |
| **Target** | `vup_any`, any of the five VUP components in the public files |
| **Features** | Poverty vulnerability (the out-of-sample score from notebook 09), CPI shock, demographics and education, program-free income source and private income, assets, housing, services, district, urban/rural. **No VUP variable**; no finance features (they contain VUP loans). |
| **Models** | District coverage baseline, logistic regression, random forest, LightGBM. Library defaults: no tuning. |
| **Validation** | The same cluster-based folds and holdout as notebook 09; the holdout is scored once |
| **Outputs** | `reports/model_results_vup_gap.csv`, `models/vup_gap_pipeline.joblib` (+ `.json`), `data/processed/vup_gap_scores.parquet`, `reports/vup_gap_by_district.csv`, `reports/figures/vup_gap_model/*.png` |

**How to read the model.** It estimates how likely a high-need household with given characteristics is to receive VUP.
- **Low predicted coverage** marks a profile that current VUP reaches relatively rarely: a *relatively underserved* profile.
- **Not causal.** The model describes coverage, not eligibility or causation. VUP has its own targeting rules (geographic phasing, categorisation, labour capacity), which this survey does not observe.
- **Out-of-sample poverty score.** The poverty score is out-of-fold for development households and comes from the development-fitted model for holdout households, so no household's own poverty label reaches its own features.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "models" / "common.py").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import precision_recall_curve, roc_curve

from src.models import common as cm
from src.models import plots as pl
from src.models import vup_gap_model as vm
from src.models import training as tr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)
vm.FIG_DIR.mkdir(parents=True, exist_ok=True)

hn = vm.load_population()
spec = vm.FEATURES
cm.assert_no_leakage(spec, "vup")
dev = hn[hn["split"].eq("development")].reset_index(drop=True)
hold = hn[hn["split"].eq("holdout")].reset_index(drop=True)
print(f"High-need households: development {len(dev):,} ({int(dev[vm.TARGET].sum())} VUP recipients, {dev['clust'].nunique()} clusters); "
      f"holdout {len(hold):,} ({int(hold[vm.TARGET].sum())} recipients, {hold['clust'].nunique()} clusters)")
print(f"Weighted coverage: development {np.average(dev[vm.TARGET], weights=dev['weight']):.3f}; holdout {np.average(hold[vm.TARGET], weights=hold['weight']):.3f}")
print("Recipients per CV fold:", dev.groupby("fold")[vm.TARGET].sum().to_dict())
print(f"Features: {len(spec.columns)}; categorical: {spec.categorical}")

## 1. Cross-validation

In [ ]:
t0 = time.time()
pipes, cols = vm.make_pipelines(spec)
oof = cm.cross_validate(pipes, dev, cols, vm.TARGET, dev["fold"])
fm = vm.fold_metrics(oof, dev, dev["fold"])
print(f"CV done in {time.time() - t0:.0f}s")
show = ["roc_auc", "pr_auc", "brier", "precision_top10", "recall_top10", "precision_top20", "recall_top20", "precision_top30", "recall_top30",
        "calibration_slope", "calibration_in_the_large"]
display(fm.groupby("model")[show].agg(["mean", "std"]).loc[vm.MODELS])
oof_metrics = pd.DataFrame([{"model": n, **cm.evaluate(dev[vm.TARGET], oof[n], dev["weight"], dev["pop_wt"])} for n in vm.MODELS])
display(oof_metrics.set_index("model")[show])
best, sel = vm.select_best(fm)
display(sel)
print(f"Selected model (best mean fold PR-AUC; simplest within 1 SE): {best}")

In [ ]:
paired = fm.pivot_table(index="fold", columns="model", values=["roc_auc", "pr_auc", "recall_top20"])
gain = pd.DataFrame({m: {metric: (paired[metric][m] - paired[metric]["district_baseline"]).agg(["mean", "min", "max"]).round(3).to_dict()
                         for metric in ["roc_auc", "pr_auc", "recall_top20"]} for m in vm.COMPLEXITY}).T
print("Gain over the district coverage baseline across the 5 folds (mean, worst and best fold):")
gain

In [ ]:
better_all = all((paired["pr_auc"][m] > paired["pr_auc"]["district_baseline"]).all() for m in vm.COMPLEXITY)
fig, axes = pl.figure("Cross-validation: models vs the district coverage baseline",
                      "High-need households, five cluster-based folds (development set); each dot is one fold, bar = mean", width=11, height=4.3, ncols=3)
for ax, metric, label in zip(axes, ["roc_auc", "pr_auc", "recall_top20"], ["ROC-AUC", "PR-AUC", "VUP recipients in top 20%"]):
    for i, m in enumerate(vm.MODELS[::-1]):
        v = fm.loc[fm["model"].eq(m), metric]
        ax.plot(v, [i] * len(v), "o", color=pl.MODEL_COLORS[m], alpha=0.75, markersize=6)
        ax.plot([v.mean()], [i], "|", color=pl.INK["primary"], markersize=14, markeredgewidth=2)
    ax.set_yticks(range(len(vm.MODELS)), [pl.MODEL_LABELS[m] for m in vm.MODELS[::-1]] if metric == "roc_auc" else [""] * len(vm.MODELS), fontsize=9)
    ax.set_title(label, fontsize=10, color=pl.INK["secondary"], loc="left")
path = pl.save(fig, vm.FIG_DIR / "cv_fold_metrics.png", top=0.74, bottom=0.14, left=0.15, wspace=0.12)
display(Image(str(path)))

## 2. Final holdout

Models refitted on all development high-need households, scored once on the holdout high-need households. Intervals come from a cluster bootstrap (500 draws, clusters resampled within strata). With few recipients in the holdout, these intervals are wide.

In [ ]:
fitted = vm.fit_on(pipes, cols, dev)
hold_pred = tr.predict(fitted, cols, hold)
hold_metrics = pd.DataFrame([{"model": n, **cm.evaluate(hold[vm.TARGET], hold_pred[n], hold["weight"], hold["pop_wt"])} for n in vm.MODELS])
hb = hold.join(hold_pred)
boot = cm.cluster_bootstrap(hb, vm.MODELS, vm.TARGET, B=500, baseline="district_baseline",
                            metrics=("roc_auc", "pr_auc", "brier", "precision_top10", "recall_top10", "precision_top20", "recall_top20",
                                     "precision_top30", "recall_top30"))
display(hold_metrics.set_index("model")[show])
ci = boot.assign(interval=lambda b: b.apply(lambda r: f"{r.ci_low:.3f}-{r.ci_high:.3f}", axis=1),
                 diff=lambda b: b.apply(lambda r: "" if pd.isna(r.get("diff_vs_baseline_ci_low")) else f"{r.diff_vs_baseline_ci_low:+.3f} to {r.diff_vs_baseline_ci_high:+.3f}", axis=1))
display(ci.pivot_table(index="score", columns="metric", values="interval", aggfunc="first").loc[vm.MODELS, ["roc_auc", "pr_auc", "recall_top20", "precision_top20"]])
print("Paired difference vs district baseline, 95% interval (share of draws where the model is better):")
d_ = ci[ci["score"].ne("district_baseline")]
display(d_.pivot_table(index="score", columns="metric", values="diff", aggfunc="first")[["roc_auc", "pr_auc", "recall_top20", "brier"]])
display(d_.pivot_table(index="score", columns="metric", values="share_of_draws_better")[["roc_auc", "pr_auc", "recall_top20", "brier"]])

In [ ]:
fig, (a1, a2) = pl.figure("Holdout ROC and precision-recall curves: predicting VUP coverage among high-need households",
                          f"Final holdout ({len(hold):,} high-need households, {int(hold[vm.TARGET].sum())} VUP recipients); AUC in brackets", width=11, height=5, ncols=2)
y, w = hold[vm.TARGET], hold["weight"]
roc, pr = {}, {}
for n in vm.MODELS:
    fpr, tpr, _ = roc_curve(y, hold_pred[n], sample_weight=w)
    p_, r_, _ = precision_recall_curve(y, hold_pred[n], sample_weight=w)
    e = hold_metrics.set_index("model").loc[n]
    roc[n], pr[n] = (fpr, tpr, e["roc_auc"]), (r_, p_, e["pr_auc"])
pl.curves(a1, "roc", roc)
pl.curves(a2, "pr", pr, baseline_rate=np.average(y, weights=w))
path = pl.save(fig, vm.FIG_DIR / "holdout_roc_pr.png", top=0.82, bottom=0.12, left=0.07, wspace=0.2)
display(Image(str(path)))

In [ ]:
fig, (ax,) = pl.figure("Calibration of predicted VUP coverage (out-of-fold, development)",
                       "Observed coverage vs mean predicted probability, by quintile of prediction (development set for sample size)", width=7, height=6)
ax.plot([0, 0.4], [0, 0.4], ":", color=pl.BASELINE)
for n in ["district_baseline", "logistic_regression", best] if best != "logistic_regression" else ["district_baseline", "logistic_regression", "lightgbm"]:
    t = cm.calibration_table(dev[vm.TARGET], oof[n], dev["weight"], n_bins=5)
    ax.plot(t["mean_predicted"], t["observed"], "o-", color=pl.MODEL_COLORS[n], linewidth=2, markersize=5, label=pl.MODEL_LABELS[n])
ax.set_xlabel("Mean predicted coverage", fontsize=9, color=pl.INK["secondary"])
ax.set_ylabel("Observed coverage", fontsize=9, color=pl.INK["secondary"])
ax.grid(axis="both", color=pl.GRID)
ax.legend(frameon=False, fontsize=8.5)
path = pl.save(fig, vm.FIG_DIR / "calibration_oof.png", top=0.86, bottom=0.1, left=0.12)
display(Image(str(path)))
display(hold_metrics.set_index("model")[["calibration_slope", "calibration_intercept", "calibration_in_the_large", "brier"]])

## 3. Who is relatively underserved?

High-need households grouped by quintile of **predicted** coverage (out-of-fold for development, development-fit model for the holdout; pooled). Quintile 1 holds the profiles current VUP reaches least often. Observed coverage in each quintile shows whether the ranking holds up.

In [ ]:
pooled = pd.concat([dev.assign(score=oof[best].to_numpy(), source="out-of-fold"),
                    hold.assign(score=hold_pred[best].to_numpy(), source="holdout")], ignore_index=True)
prof = vm.profile(pooled, "score")
prof

In [ ]:
cov_by_q = pooled.assign(q=pd.qcut(pooled["score"].rank(method="first"), 5, labels=[f"Q{i}" for i in range(1, 6)]))
tab = cm.weighted_rate_table(cov_by_q, vm.TARGET, "q", "weight", [f"Q{i}" for i in range(1, 6)])
tab.index = ["Q1 (least likely covered)", "Q2", "Q3", "Q4", "Q5 (most likely covered)"]
fig, (ax,) = pl.figure("Observed VUP coverage by quintile of predicted coverage",
                       f"High-need households, {pl.MODEL_LABELS[best]} out-of-sample predictions (development out-of-fold + holdout); 95% design-based intervals",
                       width=10, height=3.9)
pl.dot_ci(ax, tab, 100, pl.MODEL_COLORS[best])
ax.set_xlim(0, 100 * tab["ci_high"].max() * 1.2)
ax.set_xlabel("% of high-need households receiving any VUP component", fontsize=9, color=pl.INK["secondary"])
path = pl.save(fig, vm.FIG_DIR / "observed_coverage_by_predicted_quintile.png", top=0.72, bottom=0.2, left=0.3)
display(Image(str(path)))
tab

## 4. Subgroup performance

Holdout, selected model and baseline. The holdout has few recipients, so many subgroup AUCs are unreliable (fewer than 20 recipients): they are shown as missing. Development out-of-fold results, with roughly four times the sample, follow.

In [ ]:
hb["income_source"] = hb["main_livelihood_pf"]
hb["poverty_status"] = np.where(hb["poor"].eq(1), "poor", "non-poor")
cols_show = ["households", "positives", "observed_rate", "mean_predicted", "calibration_gap", "roc_auc", "roc_auc_ci_low", "roc_auc_ci_high"]
sub_hold = pd.concat({(by, n): cm.subgroup_bootstrap(hb, n, vm.TARGET, by, B=200)[cols_show]
                      for by in ["province_name", "area", "income_source", "poverty_status"] for n in [best, "district_baseline"]},
                     names=["dimension", "model", "group"])
display(sub_hold)
ob = dev.join(oof)
ob["income_source"] = ob["main_livelihood_pf"]
ob["poverty_status"] = np.where(ob["poor"].eq(1), "poor", "non-poor")
sub_oof = pd.concat({(by, n): cm.subgroup_bootstrap(ob, n, vm.TARGET, by, B=200)[cols_show]
                     for by in ["province_name", "area", "income_source", "poverty_status"] for n in [best, "district_baseline"]},
                    names=["dimension", "model", "group"])
print("Development, out-of-fold:")
sub_oof

In [ ]:
LIV = {"own_farming": "own farming", "farm_wage": "farm wage work", "nonfarm_wage": "non-farm wage work",
       "nonfarm_self_employment": "non-farm self-employment", "no_employed_member": "no employed member"}
rows = []
for by, label in [("province_name", "Province"), ("area", "Area"), ("income_source", "Income"), ("poverty_status", "Status")]:
    t = sub_oof.loc[(by, best)]
    for g, r in t.iterrows():
        rows.append({"group": f"{label}: {LIV.get(g, str(g).replace(' Province', ''))}", "households": r["households"],
                     "mean": r["roc_auc"], "ci_low": r["roc_auc_ci_low"], "ci_high": r["roc_auc_ci_high"]})
tab = pd.DataFrame(rows).set_index("group").dropna()
overall = oof_metrics.set_index("model").loc[best, "roc_auc"]
fig, (ax,) = pl.figure(f"{pl.MODEL_LABELS[best]}: ranking quality of VUP coverage within subgroups",
                       "ROC-AUC within each group, development out-of-fold, 95% cluster-bootstrap interval; dashed line = overall", width=10, height=1.9 + 0.36 * len(tab))
pl.dot_ci(ax, tab, 1, pl.MODEL_COLORS[best], label_fmt="{:.2f}", reference=overall)
ax.set_xlim(min(0.4, tab["ci_low"].min() - 0.02), 1.0)
path = pl.save(fig, vm.FIG_DIR / "subgroup_auc_oof.png", top=1 - 1.25 / (1.9 + 0.36 * len(tab)), bottom=0.5 / (1.9 + 0.36 * len(tab)), left=0.36)
display(Image(str(path)))

## 5. High-need households without VUP, by district

Descriptive: all high-need households (development and holdout pooled). Counts are weighted to the population. Coverage intervals are design-based, and districts with overlapping intervals should not be ranked against each other. "Mean predicted coverage" averages the out-of-sample predictions: a low value means the district's high-need households have profiles VUP rarely reaches.

In [ ]:
dist = vm.district_gaps(pooled, "score")
tot_unc = pooled.loc[pooled[vm.TARGET].eq(0), "weight"].sum()
print(f"High-need households without VUP (weighted): {tot_unc:,.0f} of {pooled['weight'].sum():,.0f} "
      f"({100 * tot_unc / pooled['weight'].sum():.1f}%); people: {pooled.loc[pooled[vm.TARGET].eq(0), 'pop_wt'].sum():,.0f}")
dist

In [ ]:
d_sorted = dist.sort_values("uncovered_high_need_households_weighted", ascending=True)
fig, (a1, a2) = pl.figure("High-need households without VUP, by district",
                          "Left: weighted number of uncovered high-need households. Right: observed VUP coverage of high-need households, 95% interval",
                          width=11, height=1.9 + 0.3 * len(d_sorted), ncols=2)
y_ = np.arange(len(d_sorted))
a1.barh(y_, d_sorted["uncovered_high_need_households_weighted"] / 1000, color=pl.SERIES[0], height=0.65)
a1.set_yticks(y_, [f"{d} ({p})" for d, p in zip(d_sorted["district"], d_sorted["province"])], fontsize=8)
a1.set_xlabel("Thousand households", fontsize=9, color=pl.INK["secondary"])
a1.set_ylim(-0.7, len(d_sorted) - 0.3)
tab_d = d_sorted.set_index("district").rename(columns={"observed_coverage": "mean", "coverage_ci_low": "ci_low", "coverage_ci_high": "ci_high", "high_need_sample": "households"})
pl.dot_ci(a2, tab_d.iloc[::-1], 100, pl.SERIES[2], show_n=False)
a2.set_yticks(a2.get_yticks(), [""] * len(tab_d))
a2.set_xlim(0, 100 * tab_d["ci_high"].max() * 1.1)
a2.set_xlabel("% covered", fontsize=9, color=pl.INK["secondary"])
path = pl.save(fig, vm.FIG_DIR / "uncovered_high_need_by_district.png", top=1 - 1.25 / (1.9 + 0.3 * len(d_sorted)), bottom=0.85 / (1.9 + 0.3 * len(d_sorted)), left=0.2, wspace=0.1)
display(Image(str(path)))

## 6. Sensitivity: high need = top two quintiles

In [ ]:
hn2 = vm.load_population(vm.HIGH_NEED_SENSITIVITY)
dev2, hold2 = hn2[hn2["split"].eq("development")].reset_index(drop=True), hn2[hn2["split"].eq("holdout")].reset_index(drop=True)
pipes2, cols2 = vm.make_pipelines(spec)
oof2 = cm.cross_validate(pipes2, dev2, cols2, vm.TARGET, dev2["fold"])
fm2 = vm.fold_metrics(oof2, dev2, dev2["fold"])
fitted2 = vm.fit_on(pipes2, cols2, dev2)
hp2 = tr.predict(fitted2, cols2, hold2)
hm2 = pd.DataFrame([{"model": n, **cm.evaluate(hold2[vm.TARGET], hp2[n], hold2["weight"], hold2["pop_wt"])} for n in vm.MODELS])
print(f"Top two need quintiles: development {len(dev2):,} ({int(dev2[vm.TARGET].sum())} recipients), holdout {len(hold2):,} ({int(hold2[vm.TARGET].sum())})")
display(pd.concat({"CV fold mean": fm2.groupby("model")[["roc_auc", "pr_auc", "recall_top20"]].mean(),
                   "holdout": hm2.set_index("model")[["roc_auc", "pr_auc", "recall_top20"]]}, axis=1).loc[vm.MODELS])

## 7. Save

In [ ]:
res = tr.results_table(fm, oof_metrics, hold_metrics, boot, "vup_gap_top_quintile", best)
res2 = tr.results_table(fm2, pd.DataFrame([{"model": n, **cm.evaluate(dev2[vm.TARGET], oof2[n], dev2["weight"], dev2["pop_wt"])} for n in vm.MODELS]),
                        hm2, cm.cluster_bootstrap(hold2.join(hp2), vm.MODELS, vm.TARGET, B=300, baseline="district_baseline",
                                                  metrics=("roc_auc", "pr_auc", "recall_top20", "precision_top20")), "vup_gap_top_two_quintiles", "")
pd.concat([res, res2], ignore_index=True).to_csv(vm.RESULTS_PATH, index=False)
print(f"{vm.RESULTS_PATH.relative_to(ROOT)} written")

meta = {"model": best, "population": "high-need households: distress_quintile_excl_vup == 5", "target": vm.TARGET,
        "feature_columns": cols[best], "poverty_score": "from models/poverty_vulnerability_pipeline.joblib (out-of-sample)",
        "trained_on": f"development split, high-need households: {len(dev):,} ({int(dev[vm.TARGET].sum())} VUP recipients)",
        "sample_weight": "household weight normalised to mean 1", "hyperparameters": "library defaults (no tuning)",
        "selection_rule": "highest mean fold PR-AUC; simplest within 1 SE",
        "holdout_metrics": hold_metrics.set_index("model").loc[best, ["roc_auc", "pr_auc", "brier", "precision_top20", "recall_top20"]].round(4).to_dict(),
        "interpretation": "low predicted probability = high-need profile relatively underserved by current VUP coverage; descriptive, not causal"}
tr.save_pipeline(fitted[best], vm.PIPELINE_PATH, meta)
gap = pooled[["hhid", "split", "fold", "district_name", "area", "weight", "pop_wt", "poor", "poverty_score", vm.TARGET, "score"]].rename(
    columns={"score": "predicted_vup_coverage"})
gap["uncovered"] = gap[vm.TARGET].eq(0).astype(int)
gap["underserved_rank_within_uncovered"] = gap["predicted_vup_coverage"].where(gap["uncovered"].eq(1)).rank(method="first")
gap.to_parquet(vm.GAP_SCORES_PATH, index=False)
dist.to_csv(vm.DISTRICT_PATH, index=False)
print(f"Saved {vm.PIPELINE_PATH.relative_to(ROOT)}, {vm.GAP_SCORES_PATH.relative_to(ROOT)} ({len(gap):,} high-need households), {vm.DISTRICT_PATH.relative_to(ROOT)}")